# CLIP train-split feature extraction on Colab (GPU)

Builds the two **Train-split** tensors Track V needs to mine per-attribute directions:

- `artifacts/celeba_attributes_train.pt`  — `[N_train, 40]` attribute mask, values in {0.0, 1.0}
- `artifacts/clip_image_features_train.pt` — `[N_train, 512]` L2-normalized CLIP ViT-B/32 vectors

These are **only used to mine attribute directions** (GDE §3.2, Prop. 1).  
Ranking / evaluation still uses the test-split tensors produced by `colab_extract_features.ipynb`.

**You only need two things in Google Drive before running:**
1. `celeba.zip` — zip whose top level contains `img_align_celeba/` and the CelebA `*.txt` files.
2. Nothing else — all logic is inlined here (no repo scripts are called).

Edit the paths in the **Config** cell, then `Runtime → Run all`.  
Total time on a T4 GPU: ~15–25 minutes (CelebA train split ≈ 162k images).

## 0. Config — edit these two lines

In [ ]:
# Where celeba.zip lives in your mounted Google Drive.
CELEBA_ZIP = "/content/drive/MyDrive/dlproject/celeba.zip"

# Where to copy the finished tensors in Drive (so they survive the session).
DRIVE_OUT_DIR = "/content/drive/MyDrive/dlproject"

## 1. Confirm the GPU is on

If this prints `False`, go to **Runtime → Change runtime type → T4 GPU → Save**, then rerun.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU — enable it via Runtime → Change runtime type → T4 GPU.")

## 2. Install dependencies

In [ ]:
!pip install -q transformers torchvision

## 3. Mount Drive and unpack the dataset

Recreates the layout torchvision's `CelebA` expects: `celeba/celeba/img_align_celeba/...`.  
The `ls` at the end is your checkpoint — you **must** see `img_align_celeba` and the `.txt` files, or step 4 will fail.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.exists(CELEBA_ZIP), f"celeba.zip not found at {CELEBA_ZIP} — fix CELEBA_ZIP in the Config cell."

!mkdir -p /content/celeba/celeba
!unzip -q -o "$CELEBA_ZIP" -d /content/celeba/celeba/
print("\nContents of /content/celeba/celeba (must show img_align_celeba + .txt files):")
!ls /content/celeba/celeba

## 4. Build the train-split attribute tensor

Reads `list_attr_celeba.txt` and `list_eval_partition.txt`, keeps only train-split rows (partition == 0),  
maps −1/+1 → 0.0/1.0, and saves `[N_train, 40]` float32.

In [ ]:
import torch
from pathlib import Path

CELEBA_DIR   = Path("/content/celeba/celeba")
ARTIFACTS    = Path("/content/artifacts")
ARTIFACTS.mkdir(exist_ok=True)

attr_file      = CELEBA_DIR / "list_attr_celeba.txt"
partition_file = CELEBA_DIR / "list_eval_partition.txt"

assert attr_file.exists(),      f"Missing {attr_file}"
assert partition_file.exists(), f"Missing {partition_file}"

# Build filename → partition map (0 = train, 1 = val, 2 = test).
partition = {}
with open(partition_file) as f:
    for line in f:
        parts = line.strip().split()
        partition[parts[0]] = int(parts[1])

print(f"Total images in partition file: {len(partition)}")
print(f"  train: {sum(v==0 for v in partition.values())}")
print(f"  val  : {sum(v==1 for v in partition.values())}")
print(f"  test : {sum(v==2 for v in partition.values())}")

attributes = []
train_filenames_ordered = []

with open(attr_file) as f:
    f.readline()   # line 1: total count
    f.readline()   # line 2: column header
    for line in f:
        parts    = line.strip().split()
        filename = parts[0]
        if partition.get(filename) == 0:   # train only
            attributes.append([int(x) for x in parts[1:]])
            train_filenames_ordered.append(filename)

# Map −1/+1 → 0.0/1.0  (CONTRACT §2).
attr_tensor = torch.tensor(attributes, dtype=torch.float32)
attr_tensor = (attr_tensor + 1) / 2

out_path = ARTIFACTS / "celeba_attributes_train.pt"
torch.save(attr_tensor, out_path)
print(f"\nSaved: {out_path}")
print(f"Shape: {tuple(attr_tensor.shape)}")
print(f"dtype: {attr_tensor.dtype}")
print(f"Range: [{attr_tensor.min():.1f}, {attr_tensor.max():.1f}]")
print("[OK] celeba_attributes_train.pt ready")

## 5. Build the train-split CLIP image-feature tensor

Encodes all ~162k train images with frozen CLIP ViT-B/32, L2-normalizes each row,  
and saves `[N_train, 512]` float32. Row i corresponds to `train_filenames_ordered[i]` and  
attribute row i from the tensor built in step 4.

This mirrors `clip_features.py::extract_image_features()` exactly — same model, same normalization,  
same two-step call (`vision_model` → `visual_projection`) — so train and test vectors sit in the  
same CLIP image cone and directions mined from train are valid when applied to the test DB.

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPModel
from PIL import Image
from pathlib import Path

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"   # must match clip_features.py
FEATURE_DIM     = 512
BATCH_SIZE      = 256

IMG_DIR = CELEBA_DIR / "img_align_celeba"
assert IMG_DIR.exists(), f"Missing image dir: {IMG_DIR}"

# Identical preprocessing to load_celeba_dataset() in data_loader.py.
_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275,  0.40821073],
        std =[0.26862954, 0.26130258, 0.27577711],
    ),
])


class _TrainDataset(Dataset):
    # Minimal dataset: loads images in the exact order of train_filenames_ordered
    # so row i of the feature tensor aligns with row i of the attribute tensor.
    def __init__(self, img_dir, filenames, transform):
        self.img_dir   = Path(img_dir)
        self.filenames = filenames
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img = Image.open(self.img_dir / self.filenames[idx]).convert("RGB")
        return self.transform(img)


device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print(f"Loading frozen CLIP: {CLIP_MODEL_NAME}")
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
model.eval()

dataset = _TrainDataset(IMG_DIR, train_filenames_ordered, _transform)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train images: {len(dataset)}")

feats = []
total = len(dataset)
done  = 0

with torch.no_grad():
    for pixel_values in loader:
        pixel_values  = pixel_values.to(device)
        vision_out    = model.vision_model(pixel_values=pixel_values)
        batch_feats   = model.visual_projection(vision_out.pooler_output)
        batch_feats   = F.normalize(batch_feats, p=2, dim=1)
        feats.append(batch_feats.cpu())
        done += pixel_values.shape[0]
        print(f"  encoded {done}/{total}", end="\r")

image_features = torch.cat(feats, dim=0).to(torch.float32)
print()   # newline after progress counter

# Sanity checks — mirrors _verify() in clip_features.py.
n_attrs = attr_tensor.shape[0]
n_feats = image_features.shape[0]
assert n_feats == n_attrs, (
    f"Row count mismatch: {n_feats} feature rows vs {n_attrs} attribute rows."
)
assert image_features.shape[1] == FEATURE_DIM, (
    f"Expected {FEATURE_DIM}-d vectors, got {image_features.shape[1]}."
)
norms = image_features.norm(p=2, dim=1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4), "Rows not unit-normalized."
print(f"[OK] verified: {n_feats} rows, {FEATURE_DIM}-d, unit-normalized, aligned with attributes")

out_path = ARTIFACTS / "clip_image_features_train.pt"
torch.save(image_features, out_path)
print(f"Saved: {out_path}")
print(f"Shape: {tuple(image_features.shape)}")
print(f"dtype: {image_features.dtype}")
print("[OK] clip_image_features_train.pt ready")

## 6. Save outputs to Drive and download locally

Copies both tensors to Drive (so they persist after the session ends) and triggers a browser download.  
Drop the downloaded files into your **local `Deep-Learning-Project/artifacts/`** folder.

In [ ]:
import os, shutil
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

outputs = [
    str(ARTIFACTS / "celeba_attributes_train.pt"),
    str(ARTIFACTS / "clip_image_features_train.pt"),
]

for f in outputs:
    assert os.path.exists(f), f"Missing {f} — did steps 4 and 5 finish without errors?"
    shutil.copy(f, DRIVE_OUT_DIR)
    print("copied to Drive:", f)

from google.colab import files
for f in outputs:
    files.download(f)